# Phi-4 Mini ONNX មេរៀនអំពីការហៅមុខងារដោយស្របពេល

This notebook demonstrates how to use Phi-4 Mini with ONNX Runtime GenAI for parallel function calling. Function calling allows the model to intelligently invoke external tools and APIs based on user requests.

## ទិដ្ឋភាពទូទៅ

In this tutorial, you'll learn how to:
- កំណត់ Phi-4 Mini ជាមួយ ONNX Runtime GenAI
- កំណត់ផែនរចនាសម្ព័ន្ធមុខងារ សម្រាប់កក់សំបុត្រយន្តហោះ និងសណ្ឋាគារ
- ប្រើការបង្កើតដោយណែនាំ ជាមួយវណ្ណយដ្ឋ Lark សម្រាប់លទ្ធផលដែលមានរចនាសម្ព័ន្ធ
- អនុវត្តការហៅមុខងារដោយស្របពេលសម្រាប់ស្ថានភាពកក់ធ្វើដំណើរដែលមានភាពស្មុគស្មាញ

## លក្ខខណ្ឌជាមុន

Before running this notebook, ensure you have:
- ទាញយកម៉ូឌែល Phi-4 Mini ONNX
- បានដំឡើងកញ្ចប់ `onnxruntime-genai`
- មានការយល់ដឹងមូលដ្ឋានអំពីគំនិតនៃការហៅមុខងារ


## ជំហាន 1: នាំចូលបណ្ណាល័យដែលត្រូវការ

ជាមុនសិន យើងនឹងនាំចូលបណ្ណាល័យដែលចាំបាច់សម្រាប់ការអនុវត្តការហៅមុខងាររបស់យើង។


In [1]:
import json

In [2]:
import onnxruntime_genai as og

## ជំហានទី 2: ការតំឡើង និង ការកំណត់រចនាសម្ព័ន្ធម៉ូឌែល

ឥឡូវនេះ យើងនឹងកំណត់រចនាសម្ព័ន្ធម៉ូឌែល Phi-4 Mini ONNX។ សូមប្រាកដថា​បានប្តូរផ្លូវទៅកាន់ថតម៉ូឌែលពិតរបស់អ្នក។


In [ ]:
# TODO: Replace with your actual Phi-4 Mini ONNX model path
# Download from: https://huggingface.co/microsoft/Phi-4-mini-onnx
path = 'Your phi-4-mini-onnx path'  # Update this path!

In [4]:
config = og.Config(path)

In [5]:
model = og.Model(config)

In [6]:
tokenizer = og.Tokenizer(model)
tokenizer_stream = tokenizer.create_stream()

## ជំហានទី 3: កំណត់ប៉ារ៉ាម៉ែត្រនៃការបង្កើត

កំណត់ប៉ារ៉ាម៉ែត្រនៃការបង្កើត ដើម្បីគ្រប់គ្រងរបៀបដែលម៉ូឌែលបញ្ចេញលទ្ធផល។ ការកំណត់ទាំងនេះធានាថា ចម្លើយនឹងមានលទ្ធផលកំណត់ច្បាស់ និងផ្តោតប្រសិទ្ធសម្រាប់ការហៅមុខងារ។


In [7]:
# Configure generation parameters for deterministic function calling
search_options = {}
search_options['max_length'] = 4096      # Maximum tokens to generate
search_options['temperature'] = 0.00001  # Very low temperature for deterministic output
search_options['top_p'] = 1.0            # Nucleus sampling parameter
search_options['do_sample'] = False       # Disable sampling for consistent results

## ជំហាន 4: កំណត់មុខងារដែលអាចប្រើបាន

នៅទីនេះ យើងកំណត់មុខងារដែលជំនួយការ AI របស់យើងអាចហៅបាន។ ក្នុងឧទាហរណ៍នេះ យើងមានមុខងារពាក់ព័ន្ធការធ្វើដំណើរ ពីរ៖
1. **booking_flight_tickets**: សម្រាប់កក់សំបុត្រហោះហើររវាងព្រលានអាកាស
2. **booking_hotels**: សម្រាប់កក់ទីស្នាក់នៅសណ្ឋាគារ

ការកំណត់មុខងារទាំងនេះ គឺអនុលោមទៅនឹងទ្រង់ទ្រាយ schema សម្រាប់ហៅមុខងារ របស់ OpenAI។


In [8]:
tool_list = '[{"name": "booking_flight_tickets", "description": "booking flights", "parameters": {"origin_airport_code": {"description": "The name of Departure airport code", "type": "string"}, "destination_airport_code": {"description": "The name of Destination airport code", "type": "string"}, "departure_date": {"description": "The date of outbound flight", "type": "string"}, "return_date": {"description": "The date of return flight", "type": "string"}}}, {"name": "booking_hotels", "description": "booking hotel", "parameters": {"destination": {"description": "The name of the city", "type": "string"}, "check_in_date": {"description": "The date of check in", "type": "string"}, "checkout_date": {"description": "The date of check out", "type": "string"}}}]'

## ជំហាន 5: មុខងារជំនួយសម្រាប់បង្កើតវេយ្យាករណ៍

មុខងារជំនួយទាំងនេះបំលែងការកំណត់មុខងារ​របស់​យើង​ទៅជា​មាត្រដ្ឋានវេយ្យាករណ៍ Lark, ដែលត្រូវបានប្រើដោយ ONNX Runtime GenAI សម្រាប់ការបង្កើតដែលមានការណែនាំ។ នេះធានាថាម៉ូឌែលបញ្ចូនចេញការហៅមុខងារដែលមានសុពលភាពក្នុងទ្រង់ទ្រាយ JSON ត្រឹមត្រូវ។


In [9]:
def get_lark_grammar(input_tools):
    tools_list = get_tools_list(input_tools)
    prompt_tool_input = create_prompt_tool_input(tools_list)
    if len(tools_list) == 1:
        # output = ("start: TEXT | fun_call\n" "TEXT: /[^{](.|\\n)*/\n" " fun_call: <|tool_call|> %json " + json.dumps(tools_list[0]))
        output = ("start: TEXT | fun_call\n" "TEXT: /[^{](.|\\n)*/\n" " fun_call: <|tool_call|> %json " + json.dumps(convert_tool_to_grammar_input(tools_list[0])))
        return prompt_tool_input, output
    else:
        return prompt_tool_input, "start: TEXT | fun_call \n TEXT: /[^{](.|\n)*/ \n fun_call: <|tool_call|> %json {\"anyOf\": [" + ','.join([json.dumps(tool) for tool in tools_list]) + "]}"


In [10]:
def get_tools_list(input_tools):
    # input_tools format: '[{"name": "fn1", "description": "fn details", "parameters": {"p1": {"description": "details", "type": "string"}}},
    # {"fn2": 2},{"fn3": 3}]'
    tools_list = []
    try:
        tools_list = json.loads(input_tools)
    except json.JSONDecodeError:
        raise ValueError("Invalid JSON format for tools list, expected format: '[{\"name\": \"fn1\"},{\"name\": \"fn2\"}]'")
    if len(tools_list) == 0:
        raise ValueError("Tools list cannot be empty")
    return tools_list

In [11]:
def create_prompt_tool_input(tools_list):
    tool_input = str(tools_list[0])
    for tool in tools_list[1:]:
        tool_input += ',' + str(tool)
    return tool_input

In [12]:
def convert_tool_to_grammar_input(tool):
    param_props = {}
    required_params = []
    for param_name, param_info in tool.get("parameters", {}).items():
        param_props[param_name] = {
            "type": param_info.get("type", "string"),
            "description": param_info.get("description", "")
        }
        required_params.append(param_name)
    output_schema = {
        "description": tool.get('description', ''),
        "type": "object",
        "required": ["name", "parameters"],
        "additionalProperties": False,
        "properties": {
            "name": { "const": tool["name"] },
            "parameters": {
                "type": "object",
                "properties": param_props,
                "required": required_params,
                "additionalProperties": False
            }
        }
    }
    if len(param_props) == 0:
        output_schema["required"] = ["name"]
    return output_schema

In [13]:
get_lark_grammar(tool_list)

("{'name': 'booking_flight_tickets', 'description': 'booking flights', 'parameters': {'origin_airport_code': {'description': 'The name of Departure airport code', 'type': 'string'}, 'destination_airport_code': {'description': 'The name of Destination airport code', 'type': 'string'}, 'departure_date': {'description': 'The date of outbound flight', 'type': 'string'}, 'return_date': {'description': 'The date of return flight', 'type': 'string'}}},{'name': 'booking_hotels', 'description': 'booking hotel', 'parameters': {'destination': {'description': 'The name of the city', 'type': 'string'}, 'check_in_date': {'description': 'The date of check in', 'type': 'string'}, 'checkout_date': {'description': 'The date of check out', 'type': 'string'}}}",
 'start: TEXT | fun_call \n TEXT: /[^{](.|\n)*/ \n fun_call: <|tool_call|> %json {"anyOf": [{"name": "booking_flight_tickets", "description": "booking flights", "parameters": {"origin_airport_code": {"description": "The name of Departure airport c

## ជំហានទី 6: សាកល្បងការបង្កើតវចនានុក្រម

យើងសាកល្បងអនុគមន៍បង្កើតវចនានុក្រម ដើម្បីមើលថាវាបម្លែងការកំណត់ឧបករណ៍របស់យើងទៅទ្រង់ទ្រាយដែលត្រឹមត្រូវយ៉ាងដូចម្តេច។


In [14]:
prompt_tool_input, guidance_input = get_lark_grammar(tool_list)

## ជំហាន 7: រៀបចំ System Prompt និង Generator

ឥឡូវនេះ យើង​នឹង​បង្កើត system prompt ដែលប្រាប់ម៉ូដែលអំពីឧបករណ៍ដែលមាន និងរៀបចំ generator ជាមួយប៉ារ៉ាម៉ែត្រ​សម្រាប់ការបង្កើតដែលមានការណែនាំ។


In [15]:
# Define the system prompt that introduces the assistant and its capabilities
system_prompt = "You are a helpful assistant with these tools."

In [16]:
# Format the system message with tools information
messages = f"""[{{"role": "system", "content": "{system_prompt}", "tools": "{prompt_tool_input}"}}]"""

In [17]:
# Apply chat template to format the system prompt properly
tokenizer_input_system_prompt = tokenizer.apply_chat_template(messages=messages, add_generation_prompt=False)

In [18]:
tokenizer_input_system_prompt

"<|system|>You are a helpful assistant with these tools.<|tool|>{'name': 'booking_flight_tickets', 'description': 'booking flights', 'parameters': {'origin_airport_code': {'description': 'The name of Departure airport code', 'type': 'string'}, 'destination_airport_code': {'description': 'The name of Destination airport code', 'type': 'string'}, 'departure_date': {'description': 'The date of outbound flight', 'type': 'string'}, 'return_date': {'description': 'The date of return flight', 'type': 'string'}}},{'name': 'booking_hotels', 'description': 'booking hotel', 'parameters': {'destination': {'description': 'The name of the city', 'type': 'string'}, 'check_in_date': {'description': 'The date of check in', 'type': 'string'}, 'checkout_date': {'description': 'The date of check out', 'type': 'string'}}}<|/tool|><|end|><|endoftext|>"

In [19]:
input_tokens = tokenizer.encode(tokenizer_input_system_prompt)

In [20]:
input_tokens = input_tokens[:-1]

In [21]:
system_prompt_length = len(input_tokens)

## ជំហាន 8: ចាប់ផ្តើម Generator ជាមួយការបង្កើតដែលមានការណែនាំ

ឥឡូវនេះយើងនឹងបង្កើត generator ជាមួយប៉ារ៉ាម៉ែត្រដែលបានកំណត់ ហើយអនុវត្តវេយ្យាករណ៍ Lark សម្រាប់ការ​បង្កើត​ដែលមានការណែនាំ។


In [22]:
# Create generator parameters and apply search options
params = og.GeneratorParams(model)
params.set_search_options(**search_options)

In [23]:
# Apply Lark grammar for guided generation to ensure valid function call format
params.set_guidance("lark_grammar", guidance_input)

In [24]:
generator = og.Generator(model, params)

In [25]:
generator.append_tokens(input_tokens)

## ជំហានទី 9: សាកល្បងការហៅមុខងារ​សមស្រប

ឥឡូវនេះ យើងសាកល្បងការកំណត់រចនារបស់យើងជាមួយករណីកក់ទេសចរណ៍មួយដែលស្មុគស្មាញ ហើយត្រូវការការហៅមុខងារច្រើន។


In [26]:
# Complex travel booking request that requires both flight and hotel booking
text = "book flight ticket from Beijing to Paris(using airport code) in 2025-12-04 to 2025-12-10 , then book hotel from 2025-12-04 to 2025-12-10 in Paris"

In [27]:
# Format user message and apply chat template
messages = f"""[{{"role": "user", "content": "{text}"}}]"""

# Apply Chat Template for user input
user_prompt = tokenizer.apply_chat_template(messages=messages, add_generation_prompt=True)
input_tokens = tokenizer.encode(user_prompt)
generator.append_tokens(input_tokens)

In [28]:
user_prompt

'<|user|>book flight ticket from Beijing to Paris(using airport code) in 2025-12-04 to 2025-12-10 , then book hotel from 2025-12-04 to 2025-12-10 in Paris<|end|><|assistant|>'

### មើលសំណើអ្នកប្រើដែលបានទ្រង់ទ្រាយ


### បង្កើតការហៅមុខងារ

ម៉ូដែល នឹងបង្កើតការហៅមុខងារដែលមានរចនាសម្ព័ន្ធ ដោយផ្អែកលើសំណើរបស់អ្នក។ ដោយការណែនាំក្នុងការបង្កើត, លទ្ធផលនឹងនៅក្នុងទ្រង់ទ្រាយ JSON ត្រឹមត្រូវ ដែលអាចអនុវត្តបានដោយផ្ទាល់។

**លទ្ធផលដែលរំពឹងទុក**: ម៉ូដែលគួរតែបង្កើតការហៅមុខងារ សម្រាប់៖
1. `booking_flight_tickets` ជាមួយព័ត៌មានពី Beijing (PEK) ទៅ Paris (CDG)
2. `booking_hotels` ជាមួយព័ត៌មានសណ្ឋាគារនៅ Paris

រត់សែលខាងក្រោម ដើម្បីមើលការបង្កើតក្នុងពេលពិត:


In [29]:
# Generate tokens one by one and stream the output
while not generator.is_done():
    generator.generate_next_token()
    new_token = generator.get_next_tokens()[0]
    print(tokenizer_stream.decode(new_token), end='', flush=True)

[{"name": "booking_flight_tickets", "arguments": {"origin_airport_code": "PEKK", "destination_airport_code": "CDG", "departure_date": "2025-12-04", "return_date": "2025-12-10"}}, {"name": "booking_hotels", "arguments": {"destination": "Paris", "check_in_date": "2025-12-04", "checkout_date": "2025-12-10"}}]

## សេចក្តីសន្និដ្ឋាន

🎉 **អបអរសាទរ!** អ្នកបានអនុវត្តការហៅមុខងារដោយស្រប (parallel function calling) ជាមួយ Phi-4 Mini ដោយប្រើ ONNX Runtime GenAI បានជោគជ័យ។

### អ្វីដែលអ្នកបានរៀន:

1. **ការកំណត់ម៉ូដែល**: របៀបកំណត់ Phi-4 Mini ជាមួយ ONNX Runtime GenAI
2. **ការកំណត់មុខងារ**: របៀបកំណត់រចនាសម្ព័ន្ធមុខងារ (function schemas) សម្រាប់ការហៅមុខងារ AI
3. **ការបង្កើតដោយណែនាំ**: របៀបប្រើវេយ្យាករណ៍ Lark សម្រាប់បង្កើតលទ្ធផលដែលមានរចនាសម្ព័ន្ធ
4. **ការហៅមុខងារពហុ**: របៀបដោះស្រាយសំណើស្មុគស្មាញដែលត្រូវការការហៅមុខងារច្រើន

### អត្ថប្រយោជន៍សំខាន់ៗ:

- ✅ **លទ្ធផលមានរចនាសម្ព័ន្ធ**: ការបង្កើតដោយណែនាំធានាថាការហៅមុខងារ JSON ត្រឹមត្រូវ
- ✅ **ការដំណើរការពហុ**: ដោះស្រាយការហៅមុខងារច្រើនក្នុងសំណើតែមួយ
- ✅ **ប្រសិទ្ធភាពខ្ពស់**: ONNX Runtime ផ្តល់នូវការសន្និដ្ឋានដែលបានធ្វើឲ្យមានប្រសិទ្ធភាពខ្ពស់
- ✅ **រចនាសម្ព័ន្ធអាចបត់បែនបាន**: ងាយស្រួលក្នុងការបន្ថែម ឬកែសម្រួលការកំណត់មុខងារ

### ធនធាន:

- [ឯកសារអំពី Phi-4 Mini](https://huggingface.co/microsoft/Phi-4-mini-onnx)
- [ឯកសារអំពី ONNX Runtime GenAI](https://onnxruntime.ai/docs/genai/)
- [អនុវត្តិល្អបំផុតសម្រាប់ការហៅមុខងារ](https://platform.openai.com/docs/guides/function-calling)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈដែលយើងខំប្រឹងសម្រាប់ភាពត្រឹមត្រូវ សូមយល់ឱ្យដឹងថាការបកប្រែដោយបណ្ដាញអុីទឺណិតអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវបានរួមបញ្ចូល។ ឯកសារដើមក្នុងភាសាដើមគួរត្រូវបានចាត់ទុកថាជាប្រភពឯករាជ្យសម្រាប់យោង។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងណែនាំឱ្យប្រើការបកប្រែដោយអ្នកជំនាញមនុស្ស។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយដែលកើតមានពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
